# 🇿🇦 South Africa Economic Analysis (2000–2023)

An exploratory data analysis of South Africa's macroeconomic indicators over 23 years.

**Indicators analysed:**
- GDP (total and per capita)
- GDP growth rate
- Inflation and unemployment
- Trade balance (exports vs imports)
- Government debt as % of GDP
- Foreign Direct Investment (FDI)
- Load-shedding hours and economic impact
- Electricity production

**Author:** Lamla | [GitHub](https://github.com/Lami14)

## 1. Setup & Data Loading

In [ ]:
import sys
sys.path.insert(0, '../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from load_data import load_economic_data, get_summary
from analyze_data import (
    get_gdp_analysis, get_trade_balance, get_loadshedding_impact,
    get_debt_trajectory, get_correlation_matrix, get_decade_comparison,
    print_key_findings
)
from visualize_data import generate_all_charts

sns.set_theme(style='whitegrid')
plt.rcParams['figure.facecolor'] = 'white'

SA_GREEN = '#007A4D'
SA_GOLD  = '#FFB612'
SA_RED   = '#DE3831'
SA_BLUE  = '#002395'

print('✅ Setup complete')

In [ ]:
df = load_economic_data('../data/sa_economic_data.csv')
get_summary(df)
df.head()

## 2. Key Findings Summary

In [ ]:
print_key_findings(df)

## 3. GDP Trend Analysis

In [ ]:
gdp_stats = get_gdp_analysis(df)
print('GDP Analysis Summary')
print('-' * 40)
for k, v in gdp_stats.items():
    print(f'  {k.replace("_", " ").title():<30}: {v}')

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))

ax.fill_between(df['year'], df['gdp_usd_billions'], alpha=0.15, color=SA_GREEN)
ax.plot(df['year'], df['gdp_usd_billions'], color=SA_GREEN,
        linewidth=2.5, marker='o', markersize=5)

for yr in [2009, 2020]:
    ax.axvspan(yr-0.5, yr+0.5, alpha=0.15, color=SA_RED)

peak_idx = df['gdp_usd_billions'].idxmax()
ax.annotate(f"  Peak: ${df.loc[peak_idx,'gdp_usd_billions']:.0f}bn",
            xy=(df.loc[peak_idx,'year'], df.loc[peak_idx,'gdp_usd_billions']),
            fontsize=9, color=SA_GREEN, fontweight='bold')

recession = mpatches.Patch(color=SA_RED, alpha=0.3, label='Recession Years')
ax.legend(handles=[recession])
ax.set_title('South Africa — GDP 2000–2023', fontsize=14, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('GDP (USD Billions)')
plt.xticks(df['year'], rotation=45)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
colours = [SA_GREEN if g >= 0 else SA_RED for g in df['gdp_growth_pct']]
ax.bar(df['year'], df['gdp_growth_pct'], color=colours, alpha=0.85, width=0.7)
ax.axhline(0, color='black', linewidth=0.8)
ax.axhline(df['gdp_growth_pct'].mean(), color=SA_GOLD, linestyle='--',
           linewidth=1.5, label=f"Avg: {df['gdp_growth_pct'].mean():.1f}%")
ax.set_title('South Africa — Annual GDP Growth Rate 2000–2023', fontsize=14, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('GDP Growth (%)')
ax.legend()
plt.xticks(df['year'], rotation=45)
plt.tight_layout()
plt.show()

## 4. Inflation & Unemployment

In [ ]:
fig, ax1 = plt.subplots(figsize=(13, 5))
ax1.plot(df['year'], df['inflation_pct'], color=SA_RED,
         linewidth=2, marker='o', markersize=4, label='Inflation (%)')
ax1.set_ylabel('Inflation (%)', color=SA_RED)
ax1.tick_params(axis='y', labelcolor=SA_RED)

ax2 = ax1.twinx()
ax2.plot(df['year'], df['unemployment_pct'], color=SA_BLUE,
         linewidth=2, marker='s', markersize=4, label='Unemployment (%)')
ax2.set_ylabel('Unemployment (%)', color=SA_BLUE)
ax2.tick_params(axis='y', labelcolor=SA_BLUE)

ax1.set_title('South Africa — Inflation vs Unemployment 2000–2023', fontsize=14, fontweight='bold')
ax1.set_xlabel('Year')
lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1+lines2, labels1+labels2, loc='upper left')
plt.xticks(df['year'], rotation=45)
plt.tight_layout()
plt.show()

## 5. Trade Balance

In [ ]:
trade = get_trade_balance(df)
print(f"Surplus years : {len(trade[trade['trade_status']=='Surplus'])}")
print(f"Deficit years : {len(trade[trade['trade_status']=='Deficit'])}")
trade.tail()

In [ ]:
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 8), sharex=True)

ax1.fill_between(df['year'], df['exports_usd_billions'], alpha=0.4, color=SA_GREEN, label='Exports')
ax1.fill_between(df['year'], df['imports_usd_billions'], alpha=0.4, color=SA_RED, label='Imports')
ax1.plot(df['year'], df['exports_usd_billions'], color=SA_GREEN, linewidth=1.5)
ax1.plot(df['year'], df['imports_usd_billions'], color=SA_RED, linewidth=1.5)
ax1.set_ylabel('USD Billions')
ax1.set_title('Exports vs Imports', fontsize=13, fontweight='bold')
ax1.legend()

balance = df['exports_usd_billions'] - df['imports_usd_billions']
ax2.bar(df['year'], balance, color=[SA_GREEN if x>=0 else SA_RED for x in balance], alpha=0.8)
ax2.axhline(0, color='black', linewidth=0.8)
ax2.set_ylabel('Trade Balance (USD Billions)')
ax2.set_xlabel('Year')
ax2.set_title('Trade Balance — Surplus vs Deficit', fontsize=13, fontweight='bold')
plt.xticks(df['year'], rotation=45)
plt.tight_layout()
plt.show()

## 6. Load-Shedding Economic Impact

In [ ]:
ls = get_loadshedding_impact(df)
print(f'Total load-shedding hours (2008–2023): {int(df["loadshedding_hours_per_year"].sum()):,}')
print(f'Estimated GDP lost: ~${ls["gdp_lost_estimate_bn"].sum():.0f} billion')
ls

In [ ]:
ls_df = df[df['loadshedding_hours_per_year'] > 0]
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 8), sharex=True)

ax1.bar(ls_df['year'], ls_df['loadshedding_hours_per_year'],
        color=SA_RED, alpha=0.8, width=0.6)
ax1.set_ylabel('Hours of Load-Shedding')
ax1.set_title('Load-Shedding Hours per Year', fontsize=13, fontweight='bold')

colours = [SA_GREEN if g >= 0 else SA_RED for g in ls_df['gdp_growth_pct']]
ax2.bar(ls_df['year'], ls_df['gdp_growth_pct'], color=colours, alpha=0.8, width=0.6)
ax2.axhline(0, color='black', linewidth=0.8)
ax2.set_ylabel('GDP Growth (%)')
ax2.set_xlabel('Year')
ax2.set_title('GDP Growth Rate During Load-Shedding Years', fontsize=13, fontweight='bold')
plt.xticks(ls_df['year'], rotation=45)
plt.tight_layout()
plt.show()

## 7. Government Debt

In [ ]:
debt = get_debt_trajectory(df)
print(f"Debt in 2000: {df.loc[df['year']==2000, 'government_debt_pct_gdp'].values[0]}% of GDP")
print(f"Debt in 2023: {df.loc[df['year']==2023, 'government_debt_pct_gdp'].values[0]}% of GDP")
print(f"Increase    : {df.loc[df['year']==2023, 'government_debt_pct_gdp'].values[0] - df.loc[df['year']==2000, 'government_debt_pct_gdp'].values[0]:.1f} percentage points")

In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
ax.fill_between(df['year'], df['government_debt_pct_gdp'], alpha=0.12, color=SA_RED)
ax.plot(df['year'], df['government_debt_pct_gdp'], color=SA_RED, linewidth=2.5, marker='o', markersize=4)
ax.axhline(60, color=SA_GOLD, linestyle='--', linewidth=1.5, label='60% IMF Warning Threshold')
ax.set_title('South Africa — Government Debt as % of GDP', fontsize=14, fontweight='bold')
ax.set_xlabel('Year')
ax.set_ylabel('Debt (% of GDP)')
ax.legend()
plt.xticks(df['year'], rotation=45)
plt.tight_layout()
plt.show()

## 8. Correlation Analysis

In [ ]:
corr = get_correlation_matrix(df)

labels = ['GDP Growth', 'Inflation', 'Unemployment',
          'Govt Debt %', 'FDI', 'Load-Shedding', 'Electricity']
corr.index = labels
corr.columns = labels

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, vmin=-1, vmax=1, linewidths=0.5,
            ax=ax, annot_kws={'size': 10})
ax.set_title('Economic Indicators — Pearson Correlation Matrix', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 9. Decade Comparison

In [ ]:
decades = get_decade_comparison(df)
decades

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 5))

axes[0].bar(decades['decade'], decades['avg_gdp_growth'], color=SA_GREEN, alpha=0.8)
axes[0].set_title('Avg GDP Growth by Decade', fontweight='bold')
axes[0].set_ylabel('%')

axes[1].bar(decades['decade'], decades['avg_unemployment'], color=SA_BLUE, alpha=0.8)
axes[1].set_title('Avg Unemployment by Decade', fontweight='bold')
axes[1].set_ylabel('%')

axes[2].bar(decades['decade'], decades['avg_debt_pct_gdp'], color=SA_RED, alpha=0.8)
axes[2].set_title('Avg Govt Debt by Decade', fontweight='bold')
axes[2].set_ylabel('% of GDP')

plt.suptitle('South Africa — Decade by Decade Comparison', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

## 10. Key Insights & Conclusions

### 📈 Economic Growth
- SA's **peak GDP of $416bn** was reached in 2011, driven by commodity booms and post-2008 recovery stimulus.
- Average growth of **~2.4% per year** over 23 years is below the sub-Saharan Africa average, constrained by structural issues.
- **Two recessions** — 2009 (global financial crisis) and 2020 (COVID-19) — caused significant contractions.

### ⚡ Load-Shedding Impact
- Load-shedding began in **2008** and worsened dramatically after 2019.
- **2023 was the worst year** with ~6,000 hours of outages — equivalent to 250 full days.
- There is a **strong negative correlation** between load-shedding hours and GDP growth, confirming the direct economic cost.

### 💸 Debt Crisis
- Government debt **nearly doubled** from 46% of GDP in 2000 to **84.7% in 2023**.
- The IMF's 60% warning threshold was **breached in 2020** and has not recovered.
- Rising debt limits fiscal space for infrastructure investment to resolve the energy crisis.

### 👷 Unemployment
- Unemployment **never fell below 22%** across the entire 23-year period.
- It **rose from 26.7% to 32.1%** — one of the highest unemployment rates in the world.

### 🔮 Outlook
- Resolving the energy crisis is the single biggest lever for economic recovery.
- FDI inflows remain inconsistent — investor confidence is closely tied to political stability and infrastructure reliability.
- Structural reforms in energy, labour markets and SOE management are critical to breaking the low-growth trap.